<a href="https://colab.research.google.com/github/pavankumarcode/Mastering-AI/blob/main/2__Agent__CRM_Lead_Qualifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRM Lead Qualifier Agent

## Goal

To develop an AI agent that "automatically" enriches a new sales lead (identified by an email address) by gathering publicly available company information, checking for prior engagement in the internal CRM, and assigning a preliminary qualification score.

## Context

Sales representatives often spend valuable time manually researching leads and cross-referencing internal systems before a discovery call. This process is slow, inconsistent, and often leads to a poorly prepared first interaction.

## Agent Functionality

The agent must be able to:

1. Extract Domain: Take the email address and extract the company domain name (e.g., jane@acmecorp.com → acmecorp.com).

2. Enrich Company Data: Use the domain to look up (simulated) company details like industry, size, and annual revenue.

3. Check CRM History: Search the internal (simulated) CRM for any past contact or notes associated with the lead's email.

4. Calculate Lead Score: Synthesize all gathered data to assign a qualitative priority score (e.g., High, Medium, Low).

5. Final Summary: Present a concise, actionable summary of all findings to the sales representative.

# Initialize the Agent - Get all Imports

In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

# Set up the connection to OpenAI

In [2]:
# 1. Initialize OpenAI Client
try:
    client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
except Exception as e:
    print(f"Error initializing OpenAI client: [{e}]")
    print("Please ensure your OPENAI_API_KEY is set in your environment variables.")

In [56]:
DEBUG = False

# Function to get Domain info for a specific Domain

In [55]:
"""
Based on the Domain received, gather all the information available
and return it back to the caller in JSON format.

For now, we have provided dummy data, in actual application this is where
we can connect to DB/Interet or other source to gather information.

"""

def get_domain_info(domain: str) -> str:

    print(f"Tool Called - get_domain_info : Looking up domain info for [{domain}]")

    # Mock database for testing, in real this is where we connect to other DB/Internet to get info.
    dummy_data = {
        "google.com" : {"industry": "Software" , "size": "501-1000 employees", "revenue": "$50M - $100M"},
        "amazon.com" : {"industry": "Commerce" , "size": "100-250 employees",  "revenue": "$10M - $25M"},
        "apple.com"  : {"industry": "Creaivity", "size": "5000+ employees",    "revenue": "$1B+"},
    }

    info = dummy_data.get(domain, {"industry": "Unknown", "size": "N/A", "revenue": "N/A"})

    # Return the data as a JSON string for the AI model to parse easily
    if DEBUG:
      print(info)

    return json.dumps(info)

# Function to get the User info

In [57]:
"""
Gather all the information about the user and sent it back in JSON format.

For now, we have provided dummy data, in actual application this is where
we can connect to DB/Interet or other source to gather details about the user

In production system, this could be PostgreSQL other API, Salesforce etc.
"""
def get_user_info(email: str) -> str:

    print(f"Tool Called - get_user_info : Looking up User info for emailid [{email}]")

    # Mock database for demonstration
    dummy_data = {
        "jane@google.com" : {"last_contact": "2025-11-15", "status": "Cold Lead",          "notes": "Attended webinar, no follow-up yet."},
        "bob@amazon.net"  : {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default"         : {"last_contact": "N/A",        "status": "No Record",          "notes": "New lead, first contact opportunity."},
    }

    user_info = dummy_data.get(email, dummy_data["default"])

    if DEBUG:
      print(user_info)

    return json.dumps(user_info)

# Function to calculate the Lead Score

In [58]:
"""
Calculate the Lead Score based on Domain Informaiton and User data
And classify it to High, Medium or Low Category.

"""
def calculate_lead_score(data_summary: str) -> str:

    print(f"Tool Called - calculate_lead_score : Calculating the Lead Score for User [{data_summary}]")

    data = json.loads(data_summary)
    score = "Low" # Default score

    # Simple scoring logic for demonstration
    if data["domain_info"].get("revenue", "").startswith("$1B+"):
        score = "High"
    elif data["user_info"].get("status") == "Active Opportunity":
        score = "High"
    elif data["domain_info"].get("revenue", "").startswith("$50M"):
        score = "Medium"

    return json.dumps({"lead_score": score})

# Set up a Mapping of the above functions - so it can be called

In [59]:
# Dictionary of Function Names mapped to Keys (Keys will be used as reference as Tools)
AVAILABLE_FUNCTIONS = {
    "get_domain_info"  : get_domain_info,
    "get_user_info"    : get_user_info,
    "calculate_lead_score": calculate_lead_score,
}

# Configuring the Agent's "Menu" (Tool Schema)

The Large Language Model (LLM) cannot see our Python code directly. We must describe our tools to it using a specific JSON format known as a **Schema**.

This schema tells the model:
* **What** the tool does (Description).
* **When** to use it (Context).
* **How** to use it (Parameters/Arguments).

We pass this list to the `tools` parameter in the API call later. It effectively gives the AI a "menu" of actions it can take.

In [7]:
# Provide the details of availabe Tools to our AI, it has a specific format for OpenAI
tools_schema = [

{
"type": "function",
"function":
    {
    "name": "get_domain_info",  # This is the Function name that will be called.
    "description": "Get the Companys inforamation based on its domain name.",
    "parameters":
        {
        "type": "object",
        "properties":
            {
            "domain": # This is the Input details needed for the function to work.
                {
                "type": "string",
                "description": "The company's domain name, e.g., 'google.com'"
                },
            },
        "required": ["domain"], # Mandatory parameter.
        },
    },
},

{
"type": "function",
"function":
    {
    "name": "get_user_info", # This is the Function name that will be called.
    "description": "Get the User info and notes associated with a specific email.",
    "parameters":
        {
        "type": "object",
        "properties":
            {
            "email": # This is the Input details needed for the function to work.
                {
                "type": "string",
                "description": "The full email address of the lead."
                },
            },
        "required": ["email"], # Mandatory parameter.
        },
    },
},

{
"type": "function",
"function":
    {
    "name": "calculate_lead_score", # This is the Function name that will be called.
    "description": "Calculates the Lead score (High/Medium/Low) for a user based on the Domain data and User Data.",
    "parameters":
        {
        "type": "object",
        "properties":
            {
            "data_summary": # This is the Input details needed for the function to work.
                {
                "type": "string",
                "description": "A JSON string containing the combined Domain data and User Info."
                },
            },
        "required": ["data_summary"], # Mandatory parameter.
        },
    },
},

]

# The main Agent Function

In [60]:
"""
The main Code for the Agent.

Give the System prompt and User prompt to the AI
Append the response to the main message set.
Evaluate the response, if any Tools/Function needs to called, if yes, call and append that information too
Send the complete message again to the AI
Keep doin this until AI decides to stop.

When Stopped, Send the Final Response back to the User.
"""


def run_agent(user_prompt: str):
    print(f"Agent Started working.")

    system_prompt = (
        "You are an expert CRM Lead Qualifier Agent. Your sole task is to analyze a sales lead "
        "provided via email address. You must follow these steps precisely: "
        "1. Identify the domain from the email. "
        "2. Call `get_domain_info` and `get_user_info` sequentially to gather all data. "
        "3. Combine all collected data into a single JSON object. "
        "4. Call `calculate_lead_score` with the combined JSON object. "
        "5. Finally, combine all information (Domain info, User info, and Lead score) into a single, easy-to-read summary for the user."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    collected_data = {}

    while True:
        print("AI is Thinking...")

        if DEBUG:
          print('--------------------Calling AI-----------------------------------')
          print(messages)
          print('------------------------------------------------------------------')

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto",
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        if DEBUG:
          print('------------------------------------------------------------------')
          print(response_message, messages)
          print('------------------------------------------------------------------')

        # If AI decided to call a Tool, then Call and gather info and send it to AI again.
        if response_message.tool_calls:
            tool_calls = response_message.tool_calls

            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_to_call = AVAILABLE_FUNCTIONS.get(function_name)
                function_args = json.loads(tool_call.function.arguments)

                if not function_to_call:
                    print(f"Error: Unknown function {function_name}")
                    continue

                # Execute the function
                function_result = function_to_call(**function_args)

                # Update the persistent memory (collected_data)
                if function_name == "get_domain_info":
                    collected_data["domain_info"] = json.loads(function_result)
                elif function_name == "get_user_info":
                    collected_data["user_info"] = json.loads(function_result)
                elif function_name == "calculate_lead_score":
                    # Inject the accumulated data from previous turns
                    function_args = {"data_summary": json.dumps(collected_data)}
                    function_result = function_to_call(**function_args)

                # Append the tool result as a NEW message
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": function_result
                })

                if DEBUG:
                  print(messages)

        else: # No more Tools needs to be Called.
            print("\n--- FINAL AGENT SUMMARY ---")
            print(response_message.content)
            break

# Execute and Testing

In [62]:
# Scenario 1: High-Value Lead (Large company, needs scoring)
lead_email_1 = "jane@google.com"
run_agent(f"Whats the prospects for this lead: {lead_email_1}")

Agent Started working.
AI is Thinking...
Tool Called - get_domain_info : Looking up domain info for [google.com]
Tool Called - get_user_info : Looking up User info for emailid [jane@google.com]
AI is Thinking...
Tool Called - calculate_lead_score : Calculating the Lead Score for User [{"domain_info":{"industry":"Software","size":"501-1000 employees","revenue":"$50M - $100M"},"user_info":{"last_contact":"2025-11-15","status":"Cold Lead","notes":"Attended webinar, no follow-up yet."}}]
Tool Called - calculate_lead_score : Calculating the Lead Score for User [{"domain_info": {"industry": "Software", "size": "501-1000 employees", "revenue": "$50M - $100M"}, "user_info": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."}}]
AI is Thinking...

--- FINAL AGENT SUMMARY ---
Here's the summary for the lead with the email address jane@google.com:

### Domain Information:
- **Industry:** Software
- **Company Size:** 501-1000 employees
- **Annual Re